# JSON Parsing with LangChain

LangChain provides document loaders that convert JSON and JSON Lines data into `Document` objects for RAG pipelines, letting you pull structured or semi-structured records out of nested JSON and turn them into retrievable text with metadata.

## JSON Parsing

- **`JSONLoader`** (`langchain_community.document_loaders.JSONLoader`) reads a `.json` file and uses a **`jq` schema** to extract content and build `Document` objects. It relies on the `jq` Python package (a Python binding for the `jq` command-line JSON processor), so nested/complex JSON can be navigated with jq query syntax instead of manual Python parsing.
- Key parameters:
  - `file_path`: path to the JSON (or JSONL) file.
  - `jq_schema`: a jq expression selecting the records to turn into documents (e.g., `.` for the whole file, `.messages[]` to iterate an array field, `.[].content` to pull a nested key from each item).
  - `content_key`: when each selected record is a dict, the key whose value becomes `page_content`.
  - `is_content_key_jsonl`: treat `content_key` values as JSON Lines.
  - `text_content`: whether the extracted content must be a string (`True` by default); set to `False` when extracting non-string values.
  - `metadata_func`: a callable that receives each extracted record (and the default metadata) and returns a custom metadata dict, giving full control over what gets attached to each `Document`.

## JSON Lines Parsing

- **JSON Lines (`.jsonl`)** files store one JSON object per line rather than a single JSON array/object. `JSONLoader` supports this directly via `json_lines=True`, combined with a `jq_schema` applied to each line (e.g., `.content`).
- This format is common for chat logs, streaming exports, and datasets where each line is an independent record — it avoids loading the entire file into memory as one structure and can be processed line by line.

## Custom / Manual Processing

- For full control (custom document boundaries, computed metadata, filtering, flattening deeply nested structures), load the file with Python's built-in `json` module (`json.load` / iterating lines with `json.loads`) and build `Document` objects manually — similar to the custom CSV processing approach used earlier.
- This is often preferred when:
  - The JSON structure is deeply nested and doesn't map cleanly to a single jq expression.
  - You need to merge/aggregate fields across records before creating documents.
  - You want rich, structured metadata (ids, timestamps, categories) extracted alongside `page_content`.

## When to Use Which

| Approach | Granularity | Best for |
|---|---|---|
| `JSONLoader` + `jq_schema` | Configurable via jq | Well-structured JSON, quick extraction without custom code |
| `JSONLoader` with `json_lines=True` | One document per line | JSONL logs/exports (chat data, streaming records) |
| Manual `json` parsing | Fully custom | Deeply nested/irregular JSON, custom metadata or aggregation logic |

As with CSV/Excel, the core tradeoff is between quick, declarative extraction (`jq_schema`) and full manual control — pick jq-based extraction when the JSON shape is regular and known, and fall back to manual parsing when the structure is irregular or requires custom logic to produce good retrieval units.

In [2]:
## PDf parsing

from pathlib import Path
import os

# Move to the project root, regardless of current notebook location
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory set to: {project_root}")


Working directory set to: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [3]:
##  create json files
import json
os.makedirs("data/raw/json_files",exist_ok=True)

In [4]:
# Sample nested JSON data
json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"}
            ]
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"}
            ]
        },
        {
            "id": 3,
            "name": "Carlos Ramirez",
            "role": "DevOps Engineer",
            "skills": ["Docker", "Kubernetes", "AWS", "Terraform"],
            "projects": [
                {"name": "CI/CD Migration", "status": "Completed"},
                {"name": "Infra Monitoring", "status": "In Progress"}
            ]
        },
        {
            "id": 4,
            "name": "Priya Nair",
            "role": "Product Manager",
            "skills": ["Roadmapping", "Agile", "Stakeholder Management"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Customer Portal Revamp", "status": "Planning"}
            ]
        },
        {
            "id": 5,
            "name": "Emily Chen",
            "role": "Data Scientist",
            "skills": ["Python", "PyTorch", "NLP"],
            "projects": [
                {"name": "Chatbot Evaluation", "status": "In Progress"},
                {"name": "ML Model", "status": "Completed"}
            ]
        }
    ],
    "departments": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 1000000,
            "team_size": 25
        },
        "data_science": {
            "head": "Sarah Williams",
            "budget": 750000,
            "team_size": 15
        },
        "product": {
            "head": "Priya Nair",
            "budget": 400000,
            "team_size": 8
        },
        "infrastructure": {
            "head": "Carlos Ramirez",
            "budget": 600000,
            "team_size": 10
        }
    }
}

In [5]:
json_data

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]},
  {'id': 3,
   'name': 'Carlos Ramirez',
   'role': 'DevOps Engineer',
   'skills': ['Docker', 'Kubernetes', 'AWS', 'Terraform'],
   'projects': [{'name': 'CI/CD Migration', 'status': 'Completed'},
    {'name': 'Infra Monitoring', 'status': 'In Progress'}]},
  {'id': 4,
   'name': 'Priya Nair',
   'role': 'Product Manager',
   'skills': ['Roadmapping', 'Agile', 'Stakeholder Management'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Customer Portal

In [6]:
## save json file 
with open('data/raw/json_files/company_data.json', 'w') as f:
    json.dump(json_data, f, indent=2)

In [7]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99},
    {"timestamp": "2024-01-01", "event": "user_logout", "user_id": 123},
    {"timestamp": "2024-01-02", "event": "user_login", "user_id": 456},
    {"timestamp": "2024-01-02", "event": "page_view", "user_id": 456, "page": "/products"},
    {"timestamp": "2024-01-02", "event": "page_view", "user_id": 456, "page": "/products/laptop"},
    {"timestamp": "2024-01-02", "event": "add_to_cart", "user_id": 456, "product": "Laptop", "amount": 999.99},
    {"timestamp": "2024-01-02", "event": "purchase", "user_id": 456, "amount": 999.99},
    {"timestamp": "2024-01-03", "event": "user_login", "user_id": 789},
    {"timestamp": "2024-01-03", "event": "page_view", "user_id": 789, "page": "/home"},
    {"timestamp": "2024-01-03", "event": "search", "user_id": 789, "query": "wireless mouse"},
    {"timestamp": "2024-01-03", "event": "page_view", "user_id": 789, "page": "/products/mouse"},
    {"timestamp": "2024-01-03", "event": "error", "user_id": 789, "message": "payment_declined"},
    {"timestamp": "2024-01-03", "event": "user_logout", "user_id": 789}
]

with open('data/raw/json_files/events.jsonl', 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

In [8]:
with open('data/raw/json_files/events.jsonl', 'r') as f:
    print(f.read())

{"timestamp": "2024-01-01", "event": "user_login", "user_id": 123}
{"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"}
{"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
{"timestamp": "2024-01-01", "event": "user_logout", "user_id": 123}
{"timestamp": "2024-01-02", "event": "user_login", "user_id": 456}
{"timestamp": "2024-01-02", "event": "page_view", "user_id": 456, "page": "/products"}
{"timestamp": "2024-01-02", "event": "page_view", "user_id": 456, "page": "/products/laptop"}
{"timestamp": "2024-01-02", "event": "add_to_cart", "user_id": 456, "product": "Laptop", "amount": 999.99}
{"timestamp": "2024-01-02", "event": "purchase", "user_id": 456, "amount": 999.99}
{"timestamp": "2024-01-03", "event": "user_login", "user_id": 789}
{"timestamp": "2024-01-03", "event": "page_view", "user_id": 789, "page": "/home"}
{"timestamp": "2024-01-03", "event": "search", "user_id": 789, "query": "wireless mouse"}
{"timestamp": "2024-01

In [10]:
from langchain_community.document_loaders import JSONLoader
import json

## JSONLoader with jq_shema

print("JSON loader - Extract specific fields")

# Extract employee informaiton
employee_loader = JSONLoader(
    file_path="data/raw/json_files/company_data.json",
    jq_schema='.employees[]', # jq query to extract each employee 
    text_content=False
)

employee_docs = employee_loader.load()

print(f"Loaded {len(employee_docs)} employee documents")
print(f"First employees : {employee_docs[0].page_content[:200]}...")
print(employee_docs)

JSON loader - Extract specific fields
Loaded 5 employee documents
First employees : {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status"...
[Document(metadata={'source': '/Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/json_files/company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': '/Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/json_files/company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "An

In [11]:
# Method 2: Custom JSON processing for complex structures
from typing import List
from langchain_core.documents import Document

print("\n Custom JSON Processing")

def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligent flattening and context preservation"""
    with open(filepath, 'r') as f:
        data = json.load(f)

    documents = []

    # Strategy 1: One document per employee, with full skill/project context
    for emp in data.get('employees', []):
        projects = "\n".join(
            f"- {proj['name']} (Status: {proj['status']})"
            for proj in emp.get('projects', [])
        )
        content = (
            f"Employee Profile:\n"
            f"Name: {emp['name']}\n"
            f"Role: {emp['role']}\n"
            f"Skills: {', '.join(emp['skills'])}\n\n"
            f"Projects:\n{projects}"
        )

        documents.append(Document(
            page_content=content,
            metadata={
                'source': filepath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role'],
                'skills': emp['skills'],
                'project_count': len(emp.get('projects', [])),
            }
        ))

    # Strategy 2: One document per department, so org-level questions
    # ("who heads data science?", "what's the engineering budget?") are answerable too
    for dept_name, dept in data.get('departments', {}).items():
        content = (
            f"Department: {dept_name}\n"
            f"Head: {dept['head']}\n"
            f"Budget: ${dept['budget']:,}\n"
            f"Team Size: {dept['team_size']}"
        )

        documents.append(Document(
            page_content=content,
            metadata={
                'source': filepath,
                'data_type': 'department_info',
                'department_name': dept_name,
                'head': dept['head'],
                'budget': dept['budget'],
                'team_size': dept['team_size'],
            }
        ))

    return documents


 Custom JSON Processing


In [12]:
custom_docs = process_json_intelligently("data/raw/json_files/company_data.json")
print(f"Created {len(custom_docs)} documents ({sum(d.metadata['data_type'] == 'employee_profile' for d in custom_docs)} employees, "
      f"{sum(d.metadata['data_type'] == 'department_info' for d in custom_docs)} departments)")
print("\n" + custom_docs[0].page_content)
print("\n" + custom_docs[-1].page_content)

Created 9 documents (5 employees, 4 departments)

Employee Profile:
Name: John Doe
Role: Software Engineer
Skills: Python, JavaScript, React

Projects:
- RAG System (Status: In Progress)
- Data Pipeline (Status: Completed)

Department: infrastructure
Head: Carlos Ramirez
Budget: $600,000
Team Size: 10
